# Final Project: BERT for Palestinian POS Tagging

This notebook extends Curras with a 244-sentence Palestinian Digital Campus Corpus. It trains the extended AraBERT model, trains a controlled Curras-only baseline, evaluates both models, and displays the comparison.

## 0. Load the project from GitHub
This cell makes the notebook self-contained when it is opened directly from GitHub in Google Colab.

In [9]:
from pathlib import Path

if not Path('train_and_evaluate.py').exists():
    !git clone --depth 1 https://github.com/AliShahroor/HBKU-Assignment.git /content/HBKU-Assignment
    %cd /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging
else:
    print('Project files are already available in', Path.cwd())

Project files are already available in /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging


## 1. Install dependencies

In [10]:
!pip -q install 'transformers==4.57.3' 'datasets==3.6.0' 'accelerate==1.12.0' 'scikit-learn==1.8.0' gdown
import transformers, datasets, accelerate, sklearn
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)
print('accelerate', accelerate.__version__)
print('scikit-learn', sklearn.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.57.3
datasets 3.6.0
accelerate 1.12.0
scikit-learn 1.8.0


## 2. Download the professor-provided Curras split
The Drive folder is the same public folder referenced in the original class notebook.

In [11]:
!mkdir -p data
!gdown --folder 'https://drive.google.com/drive/folders/1azGC6MLbkB3FTt5uxK1XFFdpY7X2DYsD?usp=sharing' --output data/Curras

Retrieving folder contents
Processing file 1baXnIi7Uhpj89iavTor9ZKRumO6YEqMY test.txt
Processing file 1uzWGLGxQlaOJzZ0UDQUjl4hStyiGDjzL train.txt
Processing file 1tvCfvJz9j1EgULNZKoI2YCFHeQ55Drj_ tune.txt
Processing file 1kICEVvRm9WLNk3LWGXh1tya4PPRLqP53 validation.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1baXnIi7Uhpj89iavTor9ZKRumO6YEqMY
To: /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging/data/Curras/test.txt
100% 546k/546k [00:00<00:00, 46.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1uzWGLGxQlaOJzZ0UDQUjl4hStyiGDjzL
To: /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging/data/Curras/train.txt
100% 4.12M/4.12M [00:00<00:00, 37.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tvCfvJz9j1EgULNZKoI2YCFHeQ55Drj_
To: /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging/data/Curras/tune.txt
100% 198k/198k [00:00<00:00, 32.7MB/s]
Downloadin

## 3. Build the Palestinian Digital Campus corpus
The generator uses seed 42 and recreates the extension and combined Curras files deterministically.

In [12]:
!python build_digital_campus_corpus.py

{
  "name": "Palestinian Digital Campus Corpus",
  "seed": 42,
  "sentence_count": 244,
  "token_count": 2472,
  "splits": {
    "train_sentences": 196,
    "validation_sentences": 24,
    "test_sentences": 24,
    "train_tokens": 1988,
    "validation_tokens": 230,
    "test_tokens": 254
  },
  "pos_distribution": {
    "NOUN": 1058,
    "PV": 280,
    "PUNC": 244,
    "ADV": 186,
    "IV": 183,
    "ADJ": 153,
    "PREP": 82,
    "PRON": 41,
    "NOUN_QUANT": 40,
    "NEG_PART": 32,
    "INTERROG_PART": 32,
    "NOUN_PROP": 26,
    "SUB_CONJ": 25,
    "REL_PRON": 25,
    "ENGLISHTEXT": 24,
    "CONJ": 17,
    "FUT_PART": 16,
    "CV": 8
  },
  "split_leakage_check": "passed: all sentences are unique and split overlap is zero",
  "template_split": {
    "train_template_ids": [
      19,
      14,
      10,
      26,
      22,
      6,
      5,
      12,
      11,
      15,
      9,
      25,
      29,
      21,
      16,
      27,
      1,
      13,
      18,
      2,
      17,
      

## 4. Fine-tune the extended AraBERT model
This run trains on Curras plus the new corpus and evaluates on the unchanged Curras test and the held-out Digital Campus test.

In [13]:
!python train_and_evaluate.py --epochs 2 --batch-size 12

I0000 00:00:1795505658.882686    3636 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795505663.505713    3636 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100% 4297/4297 [00:01<00:00, 3134.65 examples/s]
Map: 100% 649/649 [00:00<00:00, 3842.98 examples/s]
Map: 100% 606/606 [00:00<00:00, 3997.82 examples/s]
Map: 100% 24/24 [00:00<00:00, 2397.89 examples/s]
{'loss': 3.6949, 'grad_norm': 6.515178680419922, 'lea

## 5. Train the controlled Curras-only baseline
Only the training data changes. The checkpoint, labels, seed, hyperparameters, and test sets remain fixed.

In [14]:
!python train_and_evaluate.py --training-data curras-only --epochs 2 --batch-size 12

I0000 00:00:1795505853.345279    4512 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795505856.333256    4512 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100% 4101/4101 [00:00<00:00, 10018.00 examples/s]
Map: 100% 625/625 [00:00<00:00, 11792.09 examples/s]
Map: 100% 606/606 [00:00<00:00, 10360.02 examples/s]
Map: 100% 24/24 [00:00<00:00, 3648.01 examples/s]
{'loss': 3.7175, 'grad_norm': 6.878777503967285, '

## 6. Evaluate both models on the Palestinian challenge set

In [15]:
!python build_challenge_set.py
!python evaluate_challenge_set.py
!python evaluate_challenge_set.py --model-dir final_model_curras_only --output-stem challenge_curras_only

Wrote 10 sentences to /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging/data/Palestinian_Challenge/test.txt
I0000 00:00:1795506040.155410    5301 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795506043.076005    5301 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
{
  "sentences": 10,
  "tokens": 89,
  "correct_tokens": 82,
  "accuracy_micro_f1": 0.9213483146067416,
  "macro_f1": 0.8352240896358543,
  "top_confusions": [
    {
      "gold": "ENGLISHTEXT",
      "predicted": "NOUN",
      "count": 2
    },
    {
      "gold": "NOUN_NUM",
      "predicted": "NOUN",
      "count": 1
    },
    {
      "gold": "NOUN",
      "predicted": "ADV",
      "count": 1
    },
    {
      "gold": "INTERROG_PART",
      "predicted"

## 7. Display the final comparison
The values below are recomputed from the JSON files produced by the two training runs and challenge evaluations.

In [16]:
import json
from pathlib import Path

extended = json.loads(Path('results.json').read_text(encoding='utf-8'))
baseline = json.loads(Path('results_curras_only.json').read_text(encoding='utf-8'))
extended_challenge = json.loads(Path('challenge_results.json').read_text(encoding='utf-8'))
baseline_challenge = json.loads(Path('challenge_curras_only_results.json').read_text(encoding='utf-8'))

def scores(run, challenge):
    evaluation = run['evaluation']
    return (run['dataset']['train_tokens'],
            evaluation['curras_test']['f1_micro'],
            evaluation['digital_campus_test']['f1_micro'],
            challenge['accuracy_micro_f1'])

base = scores(baseline, baseline_challenge)
ext = scores(extended, extended_challenge)
print(f"{'Training data':<27} {'Train tokens':>12}  {'Curras F1':>9}  {'Campus F1':>9}  {'Challenge F1':>12}")
print(f"{'Curras only':<27} {base[0]:>12,}  {base[1]:>9.4f}  {base[2]:>9.4f}  {base[3]:>12.4f}")
print(f"{'Curras + extension':<27} {ext[0]:>12,}  {ext[1]:>9.4f}  {ext[2]:>9.4f}  {ext[3]:>12.4f}")
print('\nExtension minus baseline:')
print(f'Curras F1:    {ext[1] - base[1]:+.4f}')
print(f'Campus F1:    {ext[2] - base[2]:+.4f}')
print(f'Challenge F1: {ext[3] - base[3]:+.4f}')

Training data               Train tokens  Curras F1  Campus F1  Challenge F1
Curras only                       44,600     0.9401     0.9055        0.8989
Curras + extension                46,588     0.9444     0.9370        0.9213

Extension minus baseline:
Curras F1:    +0.0043
Campus F1:    +0.0315
Challenge F1: +0.0225
